# LoRP: a tiny Prolog-backed reasoning pipeline

This notebook illustrates the four stages reported for **LoRP: LLM-based Logical Reasoning via Prolog**: translation, validation, proving, and three-way majority voting. It uses prewritten candidate translations so the entire example runs without an LLM API key.

Paper: https://doi.org/10.1016/j.knosys.2025.114140

## 0. Install SWI-Prolog

Google Colab does not include SWI-Prolog by default. This installs its headless package when needed.

In [ ]:
import shutil
import subprocess

if shutil.which("swipl") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "swi-prolog-nox"],
        check=True,
    )
print(subprocess.run(["swipl", "--version"], capture_output=True, text=True, check=True).stdout.strip())

## 1. Translation candidates

In LoRP, an LLM performs this translation independently three times. Here the candidates are visible and editable. The third is valid Prolog but mistranslates the bluebird rule, illustrating why syntax validation alone cannot catch semantic errors.

In [ ]:
problem = {
    "context": [
        "Tweety is a bluebird.",
        "Every bluebird is a bird.",
        "Every bird is an animal.",
        "Every animal is warm-blooded.",
    ],
    "question": "Is Tweety warm-blooded?",
    "query": "warm_blooded(tweety)",
}

candidates = [
    {
        "id": "candidate_1",
        "program": """
bluebird(tweety).
bird(X) :- bluebird(X).
animal(X) :- bird(X).
warm_blooded(X) :- animal(X).
""",
    },
    {
        "id": "candidate_2",
        "program": """
bluebird(tweety).
bird(X) :- bluebird(X).
living_thing(X) :- bird(X).
animal(X) :- living_thing(X).
warm_blooded(X) :- animal(X).
""",
    },
    {
        "id": "candidate_3_semantic_error",
        "program": """
bluebird(tweety).
penguin(opus).
bird(X) :- penguin(X).
animal(X) :- bird(X).
warm_blooded(X) :- animal(X).
""",
    },
]

print("Context:")
for sentence in problem["context"]:
    print(" -", sentence)
print("Question:", problem["question"])

## 2–3. Validate and prove

Each candidate is written to a temporary `.pl` file. SWI-Prolog first loads it as a syntax check, then evaluates the fixed query. Arguments are passed as a list rather than interpolated into a shell command.

In [ ]:
import tempfile
from pathlib import Path

def validate_and_prove(program, query):
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "candidate.pl"
        path.write_text(program, encoding="utf-8")

        validation = subprocess.run(
            ["swipl", "-q", "-t", "halt", "-s", str(path)],
            capture_output=True, text=True,
        )
        if validation.returncode != 0:
            return {"valid": False, "answer": None, "error": validation.stderr.strip()}

        goal = f"(({query}) -> write(true) ; write(false))"
        proof = subprocess.run(
            ["swipl", "-q", "-s", str(path), "-g", goal, "-t", "halt"],
            capture_output=True, text=True,
        )
        if proof.returncode != 0:
            return {"valid": True, "answer": None, "error": proof.stderr.strip()}
        return {"valid": True, "answer": proof.stdout.strip() == "true", "error": ""}

results = []
for candidate in candidates:
    result = validate_and_prove(candidate["program"], problem["query"])
    results.append({"id": candidate["id"], **result})
results

## 4. Three-way majority vote

Only valid candidates that produced Boolean answers participate. With three runs, two matching answers form a majority.

In [ ]:
from collections import Counter

votes = [item["answer"] for item in results if item["valid"] and item["answer"] is not None]
if len(votes) < 2:
    raise RuntimeError("Too few executable translations to form a reliable vote.")
counts = Counter(votes)
answer, count = counts.most_common(1)[0]
if count <= len(votes) // 2:
    raise RuntimeError("No strict majority.")

print("Votes:", votes)
print("Final answer:", answer)
assert votes == [True, True, False]
assert answer is True

## What this captures—and what it omits

The demo makes the symbolic boundary explicit: a flexible translator proposes programs, while a deterministic interpreter decides whether each query follows. Voting tolerates the single semantic error in candidate 3. It does **not** implement the paper's complete FOL-to-extended-Prolog translation or automate natural-language translation; those require implementation details and/or an LLM not included in this API-free example.